# Redes neuronales: predicción de diabetes (Pima Indians) con datos de Kaggle
**Ejercicio:** descargar un dataset con la librería de Kaggle para Python, estudiar sus *features* y su rótulo, generar un modelo (red neuronal), contrastarlo con ejemplos de internet y analizar los resultados.

**Cómo ejecutar:** *Entorno de ejecución → Ejecutar todo*. Requiere internet (Colab lo tiene). Todo el código y la información están en este cuaderno.

**Dataset:** [`uciml/pima-indians-diabetes-database`](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) · 768 pacientes (mujeres ≥ 21 años, herencia Pima) · 8 *features* numéricas · rótulo `Outcome` (1 = diabetes, 0 = no).

**Contenido:** 1) Descarga · 2) Estudio de features y rótulo · 3) Preparación · 4) Red neuronal y comparación · 5) Evaluación · 6) Ejemplos de internet · 7) Conclusiones

## 1. Descarga con la librería de Kaggle (`kagglehub`)
Los datasets públicos se descargan sin credenciales. Si Kaggle pidiera login, ejecute `kagglehub.login()` (necesita un token de su cuenta) o suba `diabetes.csv` manualmente en la celda de respaldo.

In [ ]:
!pip -q install kagglehub
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42

In [ ]:
# === CARGA DE DATOS ===
import kagglehub
try:
    ruta = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
    csv = glob.glob(os.path.join(ruta, "**", "*.csv"), recursive=True)[0]
except Exception as e:
    print("No se pudo descargar con kagglehub:", e)
    print("Suba manualmente el archivo diabetes.csv (del dataset de Kaggle):")
    from google.colab import files
    csv = list(files.upload().keys())[0]
df = pd.read_csv(csv)
print("Archivo:", csv, "| Forma:", df.shape)
display(df.head())

## 2. Estudio de las features y del rótulo
| Feature | Significado |
|---|---|
| Pregnancies | Número de embarazos |
| Glucose | Glucosa en plasma a las 2 h de una prueba de tolerancia oral (mg/dl) |
| BloodPressure | Presión arterial diastólica (mm Hg) |
| SkinThickness | Grosor del pliegue cutáneo del tríceps (mm) |
| Insulin | Insulina sérica a las 2 h (mu U/ml) |
| BMI | Índice de masa corporal |
| DiabetesPedigreeFunction | Puntaje de antecedentes familiares de diabetes |
| Age | Edad (años) |
| **Outcome** | **Rótulo:** 1 = diabetes, 0 = no diabetes |

In [ ]:
FEATURES = [c for c in df.columns if c != "Outcome"]
print("Tipos de datos:"); print(df.dtypes.to_string())
print("\nNulos explícitos:", int(df.isna().sum().sum()))
display(df.describe().T.round(2))

In [ ]:
# Rótulo: balance de clases
conteo = df.Outcome.value_counts().sort_index()
print(conteo.to_string()); print("Prevalencia de diabetes: %.1f%%" % (100*df.Outcome.mean()))
print("Un clasificador que siempre diga 'no' acertaría %.1f%% -> línea base de accuracy." % (100*(1-df.Outcome.mean())))
conteo.rename({0:"No diabetes",1:"Diabetes"}).plot.bar(color=["#4c78a8","#e45756"], rot=0, title="Distribución del rótulo"); plt.show()

In [ ]:
# Calidad de datos: en estas columnas un 0 es biológicamente imposible -> valor faltante codificado como 0
IMPOSIBLES = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
ceros = pd.DataFrame({"ceros": (df[IMPOSIBLES]==0).sum(), "%": ((df[IMPOSIBLES]==0).mean()*100).round(1)})
display(ceros)
df_clean = df.copy(); df_clean[IMPOSIBLES] = df_clean[IMPOSIBLES].replace(0, np.nan)

In [ ]:
# Distribución de cada feature según el rótulo
fig, axs = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axs.ravel(), FEATURES):
    sns.kdeplot(data=df_clean, x=c, hue="Outcome", common_norm=False, fill=True, ax=ax, palette=["#4c78a8","#e45756"])
    ax.set_title(c)
plt.tight_layout(); plt.show()

In [ ]:
# Correlaciones entre features y con el rótulo
corr = df_clean.corr(numeric_only=True)
plt.figure(figsize=(8,6)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0); plt.title("Correlación (ceros imposibles = NaN)"); plt.show()
print("Correlación con el rótulo:"); print(corr["Outcome"].drop("Outcome").sort_values(ascending=False).round(3).to_string())

## 3. Preparación
- Los ceros imposibles pasan a `NaN` y se **imputan con la mediana** (calculada solo con el conjunto de entrenamiento, para evitar fuga de datos).
- Las features se **estandarizan** (las redes neuronales son sensibles a la escala).
- División **estratificada** 80 % entrenamiento / 20 % prueba.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df_clean[FEATURES], df_clean["Outcome"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Entrenamiento:", X_tr.shape, "| Prueba:", X_te.shape, "| Prevalencia train/test: %.3f / %.3f" % (y_tr.mean(), y_te.mean()))

## 4. Red neuronal (perceptrón multicapa) y comparación
Arquitectura base: entrada (8) → capas ocultas con activación ReLU → salida sigmoide (probabilidad de diabetes). Se entrena con el optimizador Adam y *early stopping* (se reserva el 15 % del entrenamiento para validar y se detiene cuando deja de mejorar).
Se busca la mejor configuración (capas ocultas y regularización `alpha`) con validación cruzada de 5 particiones usando AUC, y se compara con una regresión logística como línea base simple.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

def crear_red(**kw):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler()),
                     ("mlp", MLPClassifier(activation="relu", solver="adam", max_iter=2000, early_stopping=True,
                                           validation_fraction=0.15, n_iter_no_change=30, random_state=SEED, **kw))])
grid = GridSearchCV(crear_red(),
    {"mlp__hidden_layer_sizes": [(8,), (16,), (16, 8), (32, 16)], "mlp__alpha": [1e-3, 1e-2, 1e-1, 1.0]},
    cv=5, scoring="roc_auc", n_jobs=-1).fit(X_tr, y_tr)
print("Mejor configuración:", grid.best_params_, "| AUC validación cruzada: %.3f" % grid.best_score_)
red = grid.best_estimator_
base = Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler()),
                 ("lr", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
print("Capas de la red: entrada=%d, ocultas=%s, salida=1" % (len(FEATURES), grid.best_params_["mlp__hidden_layer_sizes"]))

In [ ]:
# Curva de aprendizaje de la red elegida
mlp = red.named_steps["mlp"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(mlp.loss_curve_); ax[0].set_title("Pérdida de entrenamiento"); ax[0].set_xlabel("época")
ax[1].plot(mlp.validation_scores_, color="#e45756"); ax[1].set_title("Accuracy de validación (early stopping)"); ax[1].set_xlabel("época")
plt.tight_layout(); plt.show()
print("Épocas entrenadas:", mlp.n_iter_, "| mejor accuracy de validación: %.3f" % mlp.best_validation_score_)

## 5. Evaluación en el conjunto de prueba

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
def resumen(nombre, modelo):
    p = modelo.predict_proba(X_te)[:, 1]; pred = (p >= 0.5).astype(int)
    return {"modelo": nombre, "accuracy": accuracy_score(y_te, pred), "precision": precision_score(y_te, pred),
            "recall": recall_score(y_te, pred), "f1": f1_score(y_te, pred), "auc": roc_auc_score(y_te, p)}
tabla = pd.DataFrame([resumen("Red neuronal (MLP)", red), resumen("Regresión logística", base)]).set_index("modelo").round(3)
display(tabla)
print(classification_report(y_te, red.predict(X_te), target_names=["No diabetes","Diabetes"]))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, red.predict(X_te)), display_labels=["No","Sí"]).plot(ax=ax[0], colorbar=False, cmap="Blues")
ax[0].set_title("Matriz de confusión (red, umbral 0.5)")
RocCurveDisplay.from_estimator(red, X_te, y_te, ax=ax[1], name="Red neuronal")
RocCurveDisplay.from_estimator(base, X_te, y_te, ax=ax[1], name="Reg. logística"); ax[1].plot([0,1],[0,1],"k--"); ax[1].set_title("Curva ROC")
plt.tight_layout(); plt.show()

In [ ]:
# Robustez: el conjunto de prueba es pequeño (154 casos); se repite con validación cruzada 5x3
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_red = cross_validate(red, X, y, cv=rskf, scoring=["accuracy","roc_auc"], n_jobs=-1)
cv_lr  = cross_validate(base, X, y, cv=rskf, scoring=["accuracy","roc_auc"], n_jobs=-1)
cvtab = pd.DataFrame({"Red neuronal": [cv_red["test_accuracy"].mean(), cv_red["test_accuracy"].std(), cv_red["test_roc_auc"].mean(), cv_red["test_roc_auc"].std()],
                      "Reg. logística": [cv_lr["test_accuracy"].mean(), cv_lr["test_accuracy"].std(), cv_lr["test_roc_auc"].mean(), cv_lr["test_roc_auc"].std()]},
                     index=["accuracy (media)","accuracy (desv.)","AUC (media)","AUC (desv.)"]).round(3)
display(cvtab)

In [ ]:
# En diagnóstico médico un falso negativo (no detectar la enfermedad) cuesta más: efecto del umbral
p_te = red.predict_proba(X_te)[:, 1]
filas = []
for u in [0.2, 0.3, 0.4, 0.5, 0.6]:
    pred = (p_te >= u).astype(int); tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    filas.append({"umbral": u, "recall (sensibilidad)": tp/(tp+fn), "especificidad": tn/(tn+fp), "precision": tp/max(tp+fp,1), "falsos negativos": fn})
display(pd.DataFrame(filas).set_index("umbral").round(3))

In [ ]:
# ¿Qué features usa la red? Importancia por permutación (caída del AUC al desordenar cada variable)
from sklearn.inspection import permutation_importance
imp = permutation_importance(red, X_te, y_te, scoring="roc_auc", n_repeats=30, random_state=SEED)
imp = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(color="#4c78a8", title="Importancia por permutación (caída de AUC)"); plt.show()
print(imp.sort_values(ascending=False).round(4).to_string())

## 6. Ejemplos de internet
**a) Resultados reportados en ejemplos públicos con este mismo dataset.** El repositorio [khushaljethava/diabetes_prediction](https://github.com/khushaljethava/diabetes_prediction) publica el *accuracy* de varios modelos clásicos; el tutorial de Red Hat sobre [PyTorch con este dataset](https://developers.redhat.com/learning/learn:openshift-ai:how-create-pytorch-model/resource/resources:explore-diabetes-data-set) usa el mismo archivo `diabetes.csv`. Se comparan con nuestra red (las particiones y semillas difieren, así que la comparación es orientativa).

In [ ]:
externos = pd.Series({"Regresión logística (GitHub)": 0.7792, "KNN (GitHub)": 0.7492, "SVM (GitHub)": 0.7857,
                      "Naive Bayes (GitHub)": 0.7727, "Random Forest (GitHub)": 0.8052, "Árbol de decisión (GitHub)": 0.7922,
                      "XGBoost (GitHub)": 0.7532})
externos["NUESTRA RED (prueba)"] = tabla.loc["Red neuronal (MLP)", "accuracy"]
externos["NUESTRA RED (CV 5x3)"] = cv_red["test_accuracy"].mean()
ax = externos.sort_values().plot.barh(color=["#e45756" if "NUESTRA" in i else "#9ecae9" for i in externos.sort_values().index], figsize=(8,4.5))
ax.set_xlim(0.65, 0.85); ax.set_title("Accuracy: ejemplos de internet vs. nuestra red"); plt.show()
print("Referencia clásica: Smith et al. (1988) con el algoritmo ADAP en este mismo dataset -> https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database")

**b) Casos nuevos (no están en el dataset).** La tabla siguiente contiene **cuatro perfiles ilustrativos construidos por nosotros** a partir de criterios clínicos públicos (p. ej., la ADA considera glucosa a las 2 h de 140–199 mg/dl como prediabetes y ≥ 200 mg/dl como diabetes; IMC ≥ 30 como obesidad). **No son pacientes reales.** Para cumplir el ejercicio con casos reales de internet, **reemplace o agregue filas** con ejemplos publicados (artículos, guías o tutoriales), citando la fuente en la columna `fuente`.

In [ ]:
nuevos = pd.DataFrame([
    # Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age, esperado clínico, fuente
    [0,  90, 70, 20,  80, 22.0, 0.20, 25, "riesgo bajo",    "perfil ilustrativo (glucosa normal, IMC normal)"],
    [2, 120, 75, 28, 100, 28.0, 0.40, 35, "intermedio",     "perfil ilustrativo (sobrepeso, glucosa limítrofe)"],
    [3, 150, 80, 32, 150, 31.0, 0.50, 45, "riesgo medio-alto", "perfil ilustrativo (rango de prediabetes, obesidad)"],
    [6, 190, 90, 40, 250, 38.0, 1.00, 55, "riesgo alto",    "perfil ilustrativo (glucosa cercana a 200, obesidad)"],
], columns=FEATURES + ["esperado", "fuente"])
nuevos["P(diabetes) red"]  = red.predict_proba(nuevos[FEATURES])[:, 1].round(3)
nuevos["P(diabetes) logística"] = base.predict_proba(nuevos[FEATURES])[:, 1].round(3)
nuevos["predicción red"] = np.where(nuevos["P(diabetes) red"] >= 0.5, "Diabetes", "No diabetes")
display(nuevos[["Glucose","BMI","Age","esperado","P(diabetes) red","P(diabetes) logística","predicción red"]])

## 7. Conclusiones (se calculan con los resultados de esta ejecución)

In [ ]:
r = tabla.loc["Red neuronal (MLP)"]; l = tabla.loc["Regresión logística"]
base_acc = 1 - y_te.mean()
print(f"1) Rótulo desbalanceado: ~{100*df.Outcome.mean():.0f}% diabetes. Predecir siempre 'no' daría {base_acc:.1%} de accuracy; por eso importan también recall y AUC.")
print(f"2) Calidad de datos: {int((df[IMPOSIBLES]==0).sum().sum())} ceros imposibles (sobre todo en Insulin y SkinThickness) se trataron como faltantes; ignorarlo sesga el modelo.")
print(f"3) Red neuronal en prueba: accuracy {r.accuracy:.3f}, recall {r.recall:.3f}, AUC {r.auc:.3f}. Regresión logística: accuracy {l.accuracy:.3f}, recall {l.recall:.3f}, AUC {l.auc:.3f}.")
print(f"4) En validación cruzada 5x3 la red logra accuracy {cv_red['test_accuracy'].mean():.3f} ± {cv_red['test_accuracy'].std():.3f} y AUC {cv_red['test_roc_auc'].mean():.3f}; la logística {cv_lr['test_accuracy'].mean():.3f} y {cv_lr['test_roc_auc'].mean():.3f}.")
print(f"5) Features más influyentes (permutación): {', '.join(imp.sort_values(ascending=False).index[:3])}.")
ext_min, ext_max, cv_acc = externos.iloc[:7].min(), externos.iloc[:7].max(), cv_red["test_accuracy"].mean()
pos = "dentro del mismo rango" if ext_min - 0.02 <= cv_acc <= ext_max + 0.02 else ("por encima" if cv_acc > ext_max else "por debajo")
print(f"6) Frente a los ejemplos de internet ({ext_min:.1%}-{ext_max:.1%}), el accuracy de la red en validación cruzada ({cv_acc:.1%}) queda {pos}. Con ~768 filas y 8 variables tabulares, una red no suele superar de forma clara a modelos simples.")
print("7) Con pocos datos (154 casos de prueba) las métricas varían varios puntos según la partición; hay que reportar la dispersión, no un solo número.")
print("8) Para uso médico habría que bajar el umbral para ganar sensibilidad (tabla de umbrales) y validar con datos externos; este modelo es solo un ejercicio académico.")